# Making word embeddings with HuBERT

The same analysis as `fastText.ipynb`, with HuBERT (`SZTAKI-HLT/hubert-base-cc`) in place of fastText. HuBERT gives two kinds of word vector without context, and the final table is reported for both:

- **static**: HuBERT's input embedding table, averaged over a word's subword pieces
- **isolated**: the word on its own run through the model (`[CLS] word [SEP]`), last layer averaged over its pieces

## Loading the dataset (same rows and split as fastText.ipynb)

In [1]:
from collections import Counter

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from transformers import AutoModel, AutoTokenizer

In [2]:
LABEL_COL = "Party_orientation"
SAMPLE_FRAC = 1.0  # fraction of the data to use, 1.0 for all of it

df = pd.read_pickle("./Pickles/merged_embeddings_sampled.pkl")
df = df.sample(frac=SAMPLE_FRAC, random_state=42).reset_index(drop=True)

train_df, test_df = train_test_split(df, test_size=0.1, random_state=42, stratify=df[LABEL_COL])
train_df = train_df.reset_index(drop=True)

## Loading HuBERT

In [3]:
MODEL_NAME = "SZTAKI-HLT/hubert-base-cc"
DEVICE = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE).eval()
vocab = tokenizer.get_vocab()
print(len(vocab), "vocab entries, running on", DEVICE)

W1118 11:45:04.903000 31305 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: SZTAKI-HLT/hubert-base-cc
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


32001 vocab entries, running on mps


In [4]:
TOKEN_COL, LEMMA_COL = "tokens", "lemmas"

def to_list(x):
    return x.split() if isinstance(x, str) else list(x)

pair_counts, token_counts, lemma_counts = Counter(), Counter(), Counter()
for toks, lems in zip(train_df[TOKEN_COL], train_df[LEMMA_COL]):
    toks, lems = to_list(toks), to_list(lems)
    token_counts.update(toks)
    lemma_counts.update(lems)
    pair_counts.update(zip(toks, lems))
print("unique pairs:", len(pair_counts))

unique pairs: 370954


## Check coverage

Coverage here means a word is a **single entry** in HuBERT's vocabulary. HuBERT can still embed every other word by splitting it into subword pieces, so unlike fastText.ipynb no pairs are dropped later on. HuBERT is case-sensitive and our data is lowercase, so `resolve` tries the same casings as in fastText.ipynb.

In [5]:
def resolve(w):
    """Return the casing of w that is a whole HuBERT vocab entry, or w itself if none is."""
    for cand in (w, w.capitalize(), w.upper()):
        if cand in vocab:
            return cand
    return w

def coverage(counts):
    tokens = types = 0
    for w, n in counts.items():
        if resolve(w) in vocab:
            tokens += n
            types += 1
    return tokens / sum(counts.values()), types / len(counts)

for name, counts in [("tokens", token_counts), ("lemmas", lemma_counts)]:
    t, ty = coverage(counts)
    print(f"{name:7} token coverage {t:.3f},  type coverage {ty:.3f}")

tokens  token coverage 0.827,  type coverage 0.038
lemmas  token coverage 0.901,  type coverage 0.065


## And missing words

A word counts as missing when it is not a single HuBERT vocab entry, so HuBERT has to split it into pieces. (fastText.ipynb then reloads fastText with a larger vocabulary; HuBERT's vocabulary is fixed, so there is no equivalent step.)

In [6]:
def top_missing(counts, n=30):
    miss = [(w, c) for w, c in counts.most_common() if resolve(w) not in vocab]
    return miss[:n]

print("tokens:", top_missing(token_counts))
print("lemmas:", top_missing(lemma_counts))

tokens: [('-e', 6141), ('önöknek', 4806), ('képviselőtársam', 4715), ('képviselőtársaim', 4561), ('törvényjavaslatot', 3258), ('…', 2499), ('urat', 2187), ('2010-ben', 2007), ('vitában', 1899), ('költségvetésben', 1804), ('kérdezem', 1673), ('támogassák', 1574), ('egyetlenegy', 1571), ('törvényalkotási', 1500), ('egy-egy', 1483), ('nagyon-nagyon', 1438), ('engedjék', 1335), ('összegző', 1314), ('költségvetést', 1297), ('elmondtam', 1209), ('orbán-kormány', 1170), ('kis-', 1078), ('elmondott', 1019), ('1-jétől', 961), ('merthogy', 958), ('támogatjuk', 946), ('egyetértünk', 939), ('bérek', 924), ('környezetvédelmi', 924), ('országgyűlésben', 914)]


lemmas: [('képviselőtárs', 12299), ('-e', 6141), ('felszólalás', 3705), ('…', 2499), ('javasol', 2492), ('elhangzik', 2220), ('kormányzás', 2044), ('ráadás', 2041), ('00.', 1978), ('béremelés', 1860), ('összesség', 1817), ('csökkentés', 1800), ('módosít', 1706), ('egyetlenegy', 1665), ('hangsúlyoz', 1642), ('szokik', 1641), ('vonatkozás', 1631), ('törvényalkotási', 1500), ('egy-egy', 1483), ('fenntartás', 1455), ('nagyon-nagyon', 1438), ('orbán-kormány', 1378), ('erősít', 1360), ('törvénymódosítás', 1358), ('összegző', 1319), ('hivatkozik', 1273), ('hozzátesz', 1245), ('vitatkozik', 1227), ('rezsicsökkentés', 1224), ('bizonyít', 1185)]


## Getting suffixes and counts

In [7]:
# ház -> házban works, we get -ban, but for example kéz -> kezet does not
def get_suffix(tok, lem):
    if tok != lem and tok.startswith(lem):
        return tok[len(lem):]
    return None

suffix_info = {}   # list of (token_key, lemma_key, count)
for (tok, lem), n in pair_counts.items():
    s = get_suffix(tok, lem)
    if not s:
        continue
    tk, lk = resolve(tok), resolve(lem)
    suffix_info.setdefault(s, []).append((tk, lk, n))

MIN_PAIRS = 50
MAX_SUF = 30
suffixes = sorted([s for s, v in suffix_info.items() if len(v) >= MIN_PAIRS],
                  key=lambda s: -len(suffix_info[s]))[:MAX_SUF]
print({s: len(suffix_info[s]) for s in suffixes})

# tok and lem are the words as in our df (lowercase)
# tk, lk are the same words but cased as resolve() picks for HuBERT
# suffix_info looks like: {"ban": [("házban", "ház", 12), ...], "t": [...], ...}
# suffixes can be ambiguous tho, t is accusative or past tense for example

{'t': 11121, 'nak': 6976, 'nek': 5382, 'ek': 5183, 'ok': 4904, 'ra': 4582, 'ban': 4045, 'a': 4000, 'ni': 3988, 'e': 3693, 're': 3605, 'ról': 3203, 'et': 3016, 'ben': 2982, 'ot': 2910, 'ja': 2863, 'át': 2843, 'ként': 2827, 'ét': 2740, 'k': 2682, 'en': 2605, 'okat': 2528, 'eket': 2465, 'hoz': 2436, 'ból': 2373, 'ről': 2355, 'on': 2065, 'ának': 2009, 'ének': 1856, 'ják': 1855}


## HuBERT word vectors

Every word in `suffix_info` is embedded once, in batches (the random baseline draws from all of these pairs). Padding and `[CLS]`/`[SEP]` are masked out before averaging, so a word gets the same vector whether it is embedded alone or in a padded batch.

In [8]:
@torch.no_grad()
def hubert_word_vectors(words, batch_size=256):
    """Return two dicts word -> vector: static (input embedding table) and isolated (last layer)."""
    table = model.get_input_embeddings().weight
    static, isolated = {}, {}
    for i in range(0, len(words), batch_size):
        batch = words[i:i + batch_size]
        enc = tokenizer(batch, padding=True, return_tensors="pt", return_special_tokens_mask=True)
        special = enc.pop("special_tokens_mask")
        # 1 for the word's own subword pieces, 0 for [CLS], [SEP] and padding
        mask = ((enc["attention_mask"] == 1) & (special == 0)).unsqueeze(-1).float().to(DEVICE)
        n = mask.sum(dim=1).clamp(min=1)
        enc = enc.to(DEVICE)
        stat = (table[enc["input_ids"]] * mask).sum(dim=1) / n
        iso = (model(**enc).last_hidden_state * mask).sum(dim=1) / n
        for w, s, h in zip(batch, stat.cpu().numpy(), iso.cpu().numpy()):
            static[w], isolated[w] = s, h
    return static, isolated

words = sorted({w for v in suffix_info.values() for t, l, _ in v for w in (t, l)})
static_vecs, isolated_vecs = hubert_word_vectors(words)
print(len(words), "words embedded")

306400 words embedded


## Cos similarity and offset consistency

In [9]:
# For a pair like (házban, ház), the offset is the vector token − lemma.
# It's the difference that the suffix -ban causes in the embedding space.
# If -ban means the same thing in every word, the offsets for házban − ház,
# asztalban − asztal and so on should all point in a similar direction.

def unit(M): return M / np.linalg.norm(M, axis=1, keepdims=True)

def token_lemma_cos(T, L):
    """Cosine between each token and its own lemma, averaged over all pairs."""
    return np.mean(np.sum(unit(T) * unit(L), axis=1))

def mean_pairwise_cos(D):
    """Average cosine similarity between all pairs of offset vectors, leaving out each offset with itself.

    Same value as averaging the off-diagonal of U @ U.T, without building the n x n matrix:
    the sum of all entries is |sum of rows|^2, and the diagonal adds n.
    """
    U = unit(D)
    n = len(U)
    total = U.sum(axis=0)
    return (total @ total - n) / (n * (n - 1))

def offsets(pairs, vecs):
    T = np.stack([vecs[t] for t, _ in pairs])
    L = np.stack([vecs[l] for _, l in pairs])
    return T, L, T - L

def suffix_table(vecs, seed=0):
    rng = np.random.default_rng(seed)  # random number generator
    all_pairs = [(t, l) for v in suffix_info.values() for t, l, _ in v]
    rows = []
    for s in suffixes:
        pairs = [(t, l) for t, l, _ in suffix_info[s]]
        T, L, D = offsets(pairs, vecs)
        # baseline: uses the same number of random pairs as the suffix has
        idx = rng.choice(len(all_pairs), size=len(pairs), replace=False)
        _, _, D_rand = offsets([all_pairs[i] for i in idx], vecs)
        rows.append({
            "suffix": s, "n": len(pairs),
            "cos(token, lemma)": token_lemma_cos(T, L),
            "offset consistency": mean_pairwise_cos(D),
            "random baseline": mean_pairwise_cos(D_rand),
        })
    return pd.DataFrame(rows).round(3)

### Static (input embedding table)

In [10]:
res_static = suffix_table(static_vecs)
print(res_static.to_string(index=False))

suffix     n  cos(token, lemma)  offset consistency  random baseline
     t 11121              0.858               0.188            0.046
   nak  6976              0.853               0.297            0.047
   nek  5382              0.855               0.310            0.046
    ek  5183              0.843               0.191            0.046
    ok  4904              0.863               0.218            0.046
    ra  4582              0.848               0.345            0.047
   ban  4045              0.847               0.301            0.046
     a  4000              0.864               0.222            0.047
    ni  3988              0.820               0.311            0.047
     e  3693              0.856               0.225            0.047
    re  3605              0.843               0.344            0.047
   ról  3203              0.827               0.424            0.046
    et  3016              0.853               0.233            0.047
   ben  2982              0.831   

### Isolated (last layer, word on its own)

BERT-style output vectors all point in roughly similar directions, so cosines here are high even for unrelated words. Read offset consistency against the random baseline, not on its own.

In [11]:
res_isolated = suffix_table(isolated_vecs)
print(res_isolated.to_string(index=False))

suffix     n  cos(token, lemma)  offset consistency  random baseline
     t 11121              0.972               0.245            0.121
   nak  6976              0.966               0.208            0.124
   nek  5382              0.965               0.210            0.123
    ek  5183              0.973               0.214            0.124
    ok  4904              0.975               0.223            0.123
    ra  4582              0.966               0.295            0.123
   ban  4045              0.967               0.241            0.120
     a  4000              0.975               0.291            0.125
    ni  3988              0.956               0.308            0.121
     e  3693              0.974               0.264            0.124
    re  3605              0.965               0.298            0.125
   ról  3203              0.967               0.307            0.128
    et  3016              0.967               0.296            0.123
   ben  2982              0.965   